# 📐 MinMaxScaler — Normalization — Quick Notes
---
> **Dataset:** `tips` (seaborn) | **Library:** `sklearn.preprocessing`

## 📌 Key Points
- **Normalization** → scales features to a **fixed range [0, 1]**
- Formula: `x_scaled = (x - x_min) / (x_max - x_min)`
- All values bounded between 0 and 1
- Sensitive to **outliers** (outliers compress all other values)
- Custom range: `MinMaxScaler(feature_range=(-1, 1))`
- Best for: **image processing, neural networks, distance-based models**

## 🔑 StandardScaler vs MinMaxScaler
| Feature | StandardScaler | MinMaxScaler |
|---|---|---|
| Output range | Unbounded (≈ -3 to 3) | [0, 1] fixed |
| Formula | (x-mean)/std | (x-min)/(max-min) |
| Outlier effect | Less sensitive | Very sensitive |
| Best for | Linear/SVM models | Neural nets, images |
| Preserves 0s | No | Not always |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Load & Encode
df = sns.load_dataset('tips')
df = pd.get_dummies(df, columns=['sex','smoker','day','time'], drop_first=True)
df = df.astype(int)

x = df.drop(columns=['total_bill'])
y = df['total_bill']

# 2. Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Before Scaling — tip column range:")
print(x_train['tip'].describe().round(2))

In [ ]:
# 3. Apply MinMaxScaler
mn = MinMaxScaler()

x_train_scaled = mn.fit_transform(x_train)   # fit + transform on train
x_test_scaled  = mn.transform(x_test)        # transform only on test

df_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns)
print("After MinMaxScaler — train set stats (all values in [0,1]):")
print(np.round(df_scaled.describe(), 2))

In [ ]:
# 4. Verify min=0, max=1
print("Min of each column (should be 0):")
print(df_scaled.min().round(4).head())
print("\nMax of each column (should be 1):")
print(df_scaled.max().round(4).head())

In [ ]:
# 5. Custom range: scale to [-1, 1] instead
mn_custom = MinMaxScaler(feature_range=(-1, 1))
x_custom = mn_custom.fit_transform(x_train)
df_custom = pd.DataFrame(x_custom, columns=x_train.columns)
print("Custom range [-1, 1]:")
print(df_custom.describe().round(2)[['tip','size']])

In [ ]:
# 6. Side-by-side comparison
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

ss = StandardScaler()
mn = MinMaxScaler()

tip_ss = ss.fit_transform(x_train[['tip']])
tip_mn = mn.fit_transform(x_train[['tip']])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].hist(x_train['tip'], bins=20, color='steelblue')
axes[0].set_title('Original')
axes[1].hist(tip_ss, bins=20, color='green')
axes[1].set_title('StandardScaler (mean=0, std=1)')
axes[2].hist(tip_mn, bins=20, color='orange')
axes[2].set_title('MinMaxScaler (0 to 1)')
plt.tight_layout()
plt.show()

## ⚠️ Outlier Problem
```
Original: [1, 2, 3, 4, 1000]  ← outlier!
MinMax:   [0.001, 0.001, 0.003, 0.003, 1.0]  ← all compressed!
Standard: [-0.44, -0.44, -0.44, -0.44, 1.76] ← less affected
```

## 🥇 Remember
- After scaling: **min=0, max=1** for each feature
- `mn.data_min_` and `mn.data_max_` → training min/max
- `feature_range=(-1, 1)` → useful for tanh activation (neural nets)
- Outliers destroy MinMaxScaler → use `RobustScaler` with outliers
- **Fit only on train** → same rule as StandardScaler